# 策略概述

**Agglomerative Fundamentals** 是一個形成期（$F = 252$ 交易日）配對選取策略，
以**價格行為與公司基本面的混合特徵**做階層聚類，建立配對搜尋空間：

1. 對每檔股票建構混合特徵向量：**報酬 PCA 因子載荷（5 維）⊕ log(市值) ⊕ 盈餘殖利率（$1/PE$）⊕ GICS 產業 one-hot（12 維）**
2. 三個特徵區塊**各自標準化後加權拼接**
3. 執行 **Agglomerative Clustering**（average linkage），以**每期合併距離的分位數**動態校準 `distance_threshold`
4. 以分群標籤作為分組，執行 **min-SSD 排序 + 共整合篩選**（ADF、半衰期、Hurst）選出前 `top_n` 組
5. 交易期於標準化空間重建 spread

實作模組：`strategies/formation/agglomerative_fundamentals.py`，
組合 `HDBSCAN_PCA_Loadings.py`（價格特徵萃取）與 `ssd_rolling.py`（排序與篩選）。
基本面資料來源：`dataset/fundamentals_sp500.db`（`fetch/fundamentals_yfinance.py` 產生）。


## 為何用 Agglomerative 階層聚類

- **每個點都會被分配群組**：階層聚類無「噪音點」概念，所有股票都參與分組；
  不適合成群的股票以「過小群併入 Unknown」的規則事後排除，排除比例由資料決定而非演算法先驗
- **不預設群數**：以 `distance_threshold`（每期合併距離的 75 百分位）切割 dendrogram，
  群數由當期特徵分布自然決定；避免固定 `n_clusters` 在分布變動時強迫合併異質股票
- **average linkage**：以群間平均距離合併，對混合特徵空間（連續 + one-hot）中的
  距離尺度差異較不敏感，降低單一巨型群的形成傾向


## 已知限制：基本面資料為靜態快照

市值與本益比取自 yfinance 的**單一時點快照**（非 2000–2025 逐日歷史資料）：
每個歷史形成窗都使用同一組「現在的」基本面數值，對早期窗口存在**前視偏誤**。

- 成因：免費資料源無法取得歷史逐點基本面（需 WRDS／Compustat 等付費資料商）
- 此為**已知且刻意接受**的限制，解讀本策略結果時須一併考量
- 已下市股票無快照資料，以產業中位數插補參與分群


# 參考文獻與引用對應


## 文獻 1：Hong & Hwang (2021)

> Hong, S., & Hwang, S. (2021). In search of pairs using firm fundamentals: Is pairs trading profitable? *The European Journal of Finance*, **29**(5).

**參考部分**：

- 以**企業基本面特徵**（而非純價格序列）識別潛在配對的方法論：基本面相似的公司共享現金流與估值驅動因子，其價格間的長期均衡關係有基本面基礎
- 基本面驅動配對在納入交易成本後仍具統計上顯著獲利能力的實證

**為何參考**：

- 本策略混合特徵中**基本面區塊（log 市值、盈餘殖利率 $1/PE$）**的直接依據：
  市值決定股票的規模因子暴露與流動性層級，盈餘殖利率反映估值水準——
  兩者相近的股票更可能對相同的宏觀與估值衝擊做出同向反應

📄 文獻檔案：`ref/2021-In Search of Pairs using Firm Fundamentals.pdf`


## 文獻 2：Ward (1963)

> Ward, J. H. (1963). Hierarchical grouping to optimize an objective function. *Journal of the American Statistical Association*, **58**(301), 236–244.

**參考部分**：

- 階層式聚集聚類（agglomerative hierarchical clustering）的方法框架：由每點自成一群開始，逐步合併最相近的群
- 合併過程產生的 **dendrogram（合併距離序列）**可用於事後決定分群粒度

**為何參考**：

- 本策略的分群演算法即此框架（`sklearn.AgglomerativeClustering`，average linkage）
- 「先跑完整合併路徑取得全部合併距離、再以分位數校準 `distance_threshold`」的兩段式設計，
  正是利用 dendrogram 可事後切割的特性

⚠️ **`ref/` 資料夾內無此文獻 PDF，需補充。**


## 文獻 3：Avellaneda & Lee (2010)

> Avellaneda, M., & Lee, J.-H. (2010). Statistical arbitrage in the U.S. equities market. *Quantitative Finance*, **10**(7), 761–782.

**參考部分**：

- 報酬相關矩陣的 PCA 特徵分解與因子載荷萃取（$\sqrt{\text{特徵值}}$ 加權）

**為何參考**：

- 混合特徵中**價格行為區塊（5 維報酬 PCA 因子載荷）**的計算方式出處：
  描述每檔股票對共同風險因子的暴露組合

⚠️ **`ref/` 資料夾內無此文獻 PDF，需補充。**


## 文獻 4：Gatev, Goetzmann & Rouwenhorst (2006)／Engle & Granger (1987)／Krauss, Do & Huck (2016)

> Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). Pairs trading. *RFS*, **19**(3).
> Engle, R. F., & Granger, C. W. J. (1987). Co-integration and error correction. *Econometrica*, **55**(2).
> Krauss, C., Do, X. A., & Huck, N. (2016). The profitability of pairs trading strategies. *EJOR*.

**參考部分與理由**：

- Gatev et al.：**min-SSD 距離排序**——本策略群內排序沿用此準則（經 `ssd_rolling.py` 模組複用）
  （📄 `ref/2006-Pairs Trading Performance of a Relative-Value Arbitrage Rule.pdf`）
- Engle & Granger：ADF 共整合檢定程序（⚠️ 需補充）
- Krauss et al.：半衰期（$1$–$42$ 日）與 Hurst（$H<0.5$）門檻
  （📄 `ref/2016-The profitability of pairs trading strategies distance, cointegration, and copula methols.pdf`）


# 各階段行為

策略在每個滾動形成窗（252 交易日，每 21 日滾動）內依序執行以下六個階段。


## 階段 1：混合特徵矩陣建構

對形成窗內每檔有效股票建構三區塊特徵：

**區塊 1 — 價格行為（5 維，依據：Avellaneda & Lee 2010）**：
日報酬逐股標準化後 PCA，取前 5 個主成分的因子載荷（$\sqrt{\text{特徵值}}$ 加權）。

**區塊 2 — 公司基本面（2 維，依據：Hong & Hwang 2021）**：

$$f_i = \big[\ \log(1 + \text{MarketCap}_i),\quad 1/PE_i\ \big]$$

- 缺失值以**正規化後產業的中位數**插補（產業無覆蓋則退回全域中位數）
- 插補後 winsorize（1%–99% 分位截尾）抑制極端值
- 產業標籤先經別名正規化（如 Healthcare → Health Care、Technology → Information Technology），
  統一跨資料源的 GICS 命名

**區塊 3 — 產業歸屬（12 維）**：11 個正規化 GICS 產業 + Unknown 的 one-hot 編碼。


## 階段 2：區塊分別標準化與加權拼接

$$X = \big[\ w_{price} \cdot \text{Scale}(\text{loadings}) \ \big\|\ w_{fund} \cdot \text{Scale}(f) \ \big\|\ w_{sector} \cdot \text{OneHot} \ \big], \qquad w_{price} = w_{fund} = w_{sector} = 1.0$$

**關鍵設計**：三個區塊**各自**做 `StandardScaler`，再依權重拼接——
而非對拼接後的整個矩陣做一次標準化。

**理由**：one-hot 區塊有 12 欄、基本面只有 2 欄；若做 joint 標準化，
距離計算會被欄位數多的區塊主導，欄位數少的區塊訊號被稀釋。
區塊分別標準化 + 顯式權重讓三類資訊對距離的貢獻可控。


## 階段 3：Agglomerative 分群與 distance_threshold 校準（依據：Ward 1963）

**兩段式校準**：

1. **Probe**：以 `n_clusters=1` 跑完整合併路徑（`compute_distances=True`），取得全部 $N-1$ 個合併距離
2. **校準**：取合併距離的 **75 百分位**作為門檻：

$$\text{threshold} = \text{percentile}_{75}\big(\{d_{merge}\}\big)$$

3. **分群**：`AgglomerativeClustering(n_clusters=None, distance_threshold=threshold, linkage="average")`——
   合併距離超過門檻的分支不再合併，群數由當期資料分布自然決定

**過小群處理**：成員數 < `min_cluster_size`（= 5）的群併入 `"Unknown"`，不參與配對。

**退化保護**：門檻非正時退回最大合併距離（再退回 $10^{-6}$）。


## 階段 4：群內 min-SSD 排序與統計過濾（依據：Gatev et al. 2006）

以分群標籤（`Cluster_0`, `Cluster_1`, ...）作為分組傳入 SSD 排序流程：

1. 群內對數價格 Z-Score 標準化，計算兩兩 **SSD**，升序初篩前 $\max(200,\ top\_n \times 15)$ 組
2. 協方差矩陣批次估計 OLS 對沖比例：$\beta = \text{Cov}(P'_A, P'_B)\ /\ \text{Var}(P'_B)$
3. **三道統計過濾**：

| 道次 | 檢定 | 門檻 |
| :---: | :--- | :--- |
| 1 | ADF 共整合 | $p < 0.05$ |
| 2 | OU 半衰期 | $\lambda < 0$ 且 $1 \le HL \le 42$ 日 |
| 3 | Hurst 指數 | $H < 0.50$ |

4. 通過者依 SSD 升序取前 `top_n` 組


## 階段 5：配對輸出與附加欄位

除 SSD 排序流程的標準欄位（`Hedge_Ratio`、`Spread_Mean/Std`、`Log_Mean/Std_A/B`）外，
另回填分析用欄位：

| 欄位 | 內容 | 用途 |
| :--- | :--- | :--- |
| `Sector_A` / `Sector_B` | 真實 GICS 產業 | 產業分散控管（MSR）與結果分析 |
| `Cluster_ID_A` / `Cluster_ID_B` | 分群標籤 | 配對來源群組記錄 |
| `MarketCap_A/B`、`TrailingPE_A/B` | 基本面原始值 | 報表與後續分析 |


## 階段 6：交易期的參數使用方式

排序流程不輸出 `OLS_Alpha`，交易期自然於標準化空間重建 spread：

$$P'_{i,t} = \frac{\ln P_{i,t} - \texttt{Log\_Mean}_i}{\texttt{Log\_Std}_i}, \qquad
\text{Spread}_t = P'_{A,t} - \texttt{Hedge\_Ratio} \cdot P'_{B,t}, \qquad
Z_t = \frac{\text{Spread}_t - \texttt{Spread\_Mean}}{\texttt{Spread\_Std}}$$

形成期統計量整個交易期凍結不變（無前視——但注意基本面快照的前視限制另計，見「已知限制」一節）。
交易決策細節見 `trading/zscore_trading.ipynb`；
本策略的形成期配對另供 DRL 門檻選擇式交易端使用（見 `trading/drl_threshold_trading.ipynb`）。


# 參數總表

| 參數 | 值 | 說明 |
| :--- | :---: | :--- |
| 形成窗長度 $F$ | 252 交易日 | 約一年 |
| 滾動步長 | 21 交易日 | 約一個月 |
| `top_n` | 網格 [1, 3, 5, 10, 20] | 每期選取配對數 |
| `pca_n_components` | 5 | 價格區塊因子數 |
| `price/fundamentals/sector_weight` | 各 1.0 | 三區塊權重 |
| `agg_linkage` | average | 群間平均距離合併 |
| `agg_threshold_percentile` | 75.0 | distance_threshold 校準分位數 |
| `min_cluster_size` | 5 | 過小群併入 Unknown |
| `adf_pvalue_threshold` | 0.05 | ADF 檢定顯著水準 |
| 半衰期範圍 | $[1,\ 42]$ 日 | 交易期 126 日 ÷ 3 |
| Hurst 上限 | 0.50 | 均值回歸判準 |
| `fundamentals_db_path` | `dataset/fundamentals_sp500.db` | 基本面快照（靜態，見已知限制） |
